In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [2]:
HVFHV_PATH = "hdfs:///tlc/raw/hvfhv"
ZONES_PART_PATH = "hdfs:///tlc/raw/zones/part-00000-f8135e00-968c-4f34-a8fc-7e4182eaa810-c000.snappy.parquet"
ZONES_TAXI_PATH = "hdfs:///tlc/raw/zones/taxi_zones.parquet"

STAGING_PATH = "hdfs:///tlc/silver"
OUTPUT_PATH = f"{STAGING_PATH}/trip_clean_data"
REJECTS_PATH = f"{STAGING_PATH}/trip_rejected_data"
SCHEMA_LOG_DIR = f"{STAGING_PATH}/_schema_logs"

PEAK_HOURS = {7, 8, 9, 16, 17, 18, 19}
VALID_LICENSES = {"HV0002", "HV0003", "HV0004", "HV0005"}

DEV_MODE = True
DEV_SAMPLE_SIZE = 500_000
SAMPLE_SEED = 42

In [3]:
def get_spark() -> SparkSession:
    return (
        SparkSession.builder
        .appName("TLC-HVFHV")
        .master("local[4]")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.memory", "6g")
        .config("spark.executor.memory", "6g")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )

In [4]:
def read_raw_trips(spark: SparkSession):
    df = spark.read.parquet(HVFHV_PATH)

    if DEV_MODE:
        df = (
            df
            .sample(
                withReplacement=False,
                fraction=0.03,
                seed=SAMPLE_SEED
            )
            .limit(DEV_SAMPLE_SIZE)
        )

    return df

def read_zones(spark: SparkSession):
    return spark.read.parquet(ZONES_PART_PATH)


In [5]:
def log_schema(df, label: str):
    print(f"\n{'=' * 70}\nSCHEMA: {label}\n{'=' * 70}")
    df.printSchema()

In [6]:
def standardize_and_cast(df):
    """
    - Rename to consistent snake_case (source is already snake_case, but I will
      normalize a couple of names so they read cleanly in the warehouse).
    - Explicitly cast every column to its intended type. The raw parquet
      already carries correct physical types, but casting here makes the
      contract explicit and protects against future schema drift in new
      monthly files.
    """
    df = (
        df
        .withColumnRenamed("hvfhs_license_num", "license_num")
        .withColumnRenamed("PULocationID", "pu_location_id")
        .withColumnRenamed("DOLocationID", "do_location_id")
    )

    df = df.select(
        F.col("license_num").cast(T.StringType()),
        F.col("dispatching_base_num").cast(T.StringType()),
        F.col("originating_base_num").cast(T.StringType()),
        F.col("request_datetime").cast(T.TimestampType()),
        F.col("on_scene_datetime").cast(T.TimestampType()),
        F.col("pickup_datetime").cast(T.TimestampType()),
        F.col("dropoff_datetime").cast(T.TimestampType()),
        F.col("pu_location_id").cast(T.IntegerType()),
        F.col("do_location_id").cast(T.IntegerType()),
        F.col("trip_miles").cast(T.DoubleType()),
        F.col("trip_time").cast(T.LongType()),
        F.col("base_passenger_fare").cast(T.DoubleType()),
        F.col("tolls").cast(T.DoubleType()),
        F.col("bcf").cast(T.DoubleType()),
        F.col("sales_tax").cast(T.DoubleType()),
        F.col("congestion_surcharge").cast(T.DoubleType()),
        F.col("airport_fee").cast(T.DoubleType()),
        F.col("tips").cast(T.DoubleType()),
        F.col("driver_pay").cast(T.DoubleType()),
        F.col("shared_request_flag").cast(T.StringType()),
        F.col("shared_match_flag").cast(T.StringType()),
        F.col("access_a_ride_flag").cast(T.StringType()),
        F.col("wav_request_flag").cast(T.StringType()),
        F.col("wav_match_flag").cast(T.StringType()),
    )
    return df

In [7]:
def handle_nulls(df):
    """
    From EDA: on_scene_datetime and originating_base_num are null for the
    same ~5.2M rows (trips with no dispatch match event, e.g. flat-fare /
    non-dispatch flows). These nulls are legitimate business states, not
    data quality errors, so I will handle them as following:
      - originating_base_num: fill with 'UNKNOWN' so it's join/group-safe.
      - on_scene_datetime: leave as NULL (a real absence of an event), but
        add a boolean flag so downstream consumers don't have to
        re-derive "was this trip ever on-scene-matched".
      - Any of the numeric fare columns that come in null are filled with 0.0 rather than
        dropped, since a missing surcharge/fee most often means "$0", not
        "unknown".
    """
    fare_cols = [
        "base_passenger_fare", "tolls", "bcf", "sales_tax",
        "congestion_surcharge", "airport_fee", "tips", "driver_pay",
    ]

    df = df.withColumn(
        "originating_base_num",
        F.coalesce(F.col("originating_base_num"), F.lit("UNKNOWN"))
    )
    df = df.withColumn(
        "was_on_scene_matched",
        F.col("on_scene_datetime").isNotNull()
    )
    df = df.fillna(0.0, subset=fare_cols)

    # Any row still missing a hard-required field (can't analyze a trip
    # without these) is unrecoverable — drop it.
    required = [
        "pickup_datetime", "dropoff_datetime",
        "pu_location_id", "do_location_id", "trip_miles", "trip_time",
    ]
    df = df.dropna(subset=required)
    return df

In [8]:
def remove_duplicates(df):
    """
    Exact full-row duplicates (byte-identical records, e.g. from a
    re-ingested/overlapping file) are dropped outright.

    Business-key duplicates (same license, dispatch base, pickup/dropoff
    time and locations) are then deduplicated with a window function,
    keeping the row with the latest request_datetime as the "true" record —
    this handles the case of the same trip appearing twice with a small
    field difference (e.g. a corrected fare).
    """
    before = df.count()  
    df = df.dropDuplicates()

    business_key = [
        "license_num", "dispatching_base_num",
        "pickup_datetime", "dropoff_datetime",
        "pu_location_id", "do_location_id",
    ]
    w = Window.partitionBy(*business_key).orderBy(F.col("request_datetime").desc())
    df = (
        df.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )
    after = df.count()
    print(f"Duplicate removal: {before:,} -> {after:,} rows "
          f"({before - after:,} duplicates dropped)")
    return df

In [9]:
def clean_and_filter(df):
    """
    Business-rule cleansing:
      - dropoff must be >= pickup (EDA found 0 violations in this batch,
        but the rule stays as a safety net for future files)
      - trip_miles, trip_time, driver_pay must be non-negative
      - license_num must be one of the four known HVFHV operators (EDA
        shows only 2 are present in this particular month's file)
      - Y/N flag columns standardized to booleans (EDA confirms every flag
        column is exactly 2 distinct values, Y/N, with 0 nulls — no
        unexpected codes to handle)

    Rows failing these checks are written out to REJECTS_PATH instead of
    silently dropped.
    """
    flag_cols = [
        "shared_request_flag", "shared_match_flag",
        "access_a_ride_flag", "wav_request_flag", "wav_match_flag",
    ]
    for c in flag_cols:
        df = df.withColumn(c, F.when(F.col(c) == "Y", True)
                                .when(F.col(c) == "N", False)
                                .otherwise(None))

    valid_mask = (
        (F.col("dropoff_datetime") >= F.col("pickup_datetime"))
        & (F.col("trip_miles") >= 0)
        & (F.col("trip_time") >= 0)
        & (F.col("driver_pay") >= 0)
        & (F.col("license_num").isin(list(VALID_LICENSES)))
    )

    clean_df = df.filter(valid_mask)
    reject_df = df.filter(~valid_mask)

    reject_count = reject_df.count()
    if reject_count > 0:
        print(f"Cleansing: writing {reject_count:,} rejected rows to {REJECTS_PATH}")
        (reject_df.write.mode("overwrite").parquet(REJECTS_PATH))
    else:
        print("Cleansing: no rows failed validation rules")

    return clean_df


In [10]:
def add_derived_fields(df):
    df = (
        df
        .withColumn(
            "trip_duration_sec",
            F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")
        )
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
        .withColumn("pickup_hour", F.hour("pickup_datetime"))
        .withColumn("dropoff_hour", F.hour("dropoff_datetime"))
        # 1 = Sunday ... 7 = Saturday (Spark's dayofweek convention)
        .withColumn("day_of_week", F.dayofweek("pickup_datetime"))
        .withColumn("day", F.dayofmonth("pickup_datetime"))
        .withColumn("month", F.month("pickup_datetime"))
        .withColumn("year", F.year("pickup_datetime"))
        .withColumn("is_weekend", F.col("day_of_week").isin([1, 7]))
        .withColumn("is_peak_hour", F.col("pickup_hour").isin(list(PEAK_HOURS)))
        .withColumn(
            "total_fare",
            F.col("base_passenger_fare") + F.col("tolls") + F.col("bcf")
            + F.col("sales_tax") + F.col("congestion_surcharge")
            + F.col("airport_fee") + F.col("tips")
        )
    )
    return df

In [11]:
def enrich_with_zones(df, zones_df):
    """
    Note from EDA: Borough has 8 distinct values in the zones table,
    including 'Unknown' and (per the real TLC lookup) an 'N/A' bucket for
    a couple of non-standard LocationIDs, these are legitimate zone
    categories, not join failures, so we leave them as-is rather than
    filtering them out.
    """
    zones_pu = zones_df.select(
        F.col("LocationID").alias("pu_location_id"),
        F.col("Borough").alias("pu_borough"),
        F.col("Zone").alias("pu_zone"),
        F.col("service_zone").alias("pu_service_zone"),
    )
    zones_do = zones_df.select(
        F.col("LocationID").alias("do_location_id"),
        F.col("Borough").alias("do_borough"),
        F.col("Zone").alias("do_zone"),
        F.col("service_zone").alias("do_service_zone"),
    )

    df = df.join(F.broadcast(zones_pu), on="pu_location_id", how="left")
    df = df.join(F.broadcast(zones_do), on="do_location_id", how="left")
    return df

In [12]:
def check_referential_integrity(df, zones_df):
    """
    Use a left-anti join rather than a full join, so it
    only returns unmatched rows and stays cheap.
    """
    zone_ids = zones_df.select(F.col("LocationID").alias("id")).distinct()

    orphan_pu = df.select("pu_location_id").distinct().join(
        zone_ids, F.col("pu_location_id") == F.col("id"), "left_anti"
    )
    orphan_do = df.select("do_location_id").distinct().join(
        zone_ids, F.col("do_location_id") == F.col("id"), "left_anti"
    )

    pu_count = orphan_pu.count() 
    do_count = orphan_do.count() 

    print(f"\nReferential integrity check: "
          f"{pu_count} pickup location IDs and {do_count} dropoff location "
          f"IDs not found in the zones table.")
    if pu_count > 0:
        orphan_pu.show(20, truncate=False)
    if do_count > 0:
        orphan_do.show(20, truncate=False)



In [13]:
def run_sql_and_aggregation_demo(spark: SparkSession, df):
    df.createOrReplaceTempView("fact_trip_clean")

    print("\n--- Spark SQL demo: trips & avg fare by borough (top 10) ---")
    spark.sql("""
        SELECT pu_borough,
               pu_service_zone,
               COUNT(*)              AS trip_count,
               ROUND(AVG(total_fare), 2)   AS avg_fare,
               ROUND(AVG(trip_duration_sec), 0) AS avg_duration_sec
        FROM fact_trip_clean
        GROUP BY pu_borough, pu_service_zone
        ORDER BY trip_count DESC
        LIMIT 10
    """).show(truncate=False)

    print("\n--- DataFrame API aggregation demo: trips by day_of_week / peak hour ---")
    (
        df.groupBy("day_of_week", "is_peak_hour")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.avg("driver_pay"), 2).alias("avg_driver_pay"),
        )
        .orderBy("day_of_week", "is_peak_hour")
        .show(20, truncate=False)
    )


In [14]:
def write_output(df):
    """
    Write the cleaned dataset as Parquet partitioned by year/month/day.
    """
    (
        df
        .repartition(4, "year", "month", "day")
        .write
        .mode("overwrite")
        .partitionBy("year", "month", "day")
        .parquet(OUTPUT_PATH)
    )

    print(f"\nWrote clean dataset to {OUTPUT_PATH}")

In [15]:
spark = get_spark()
spark.sparkContext.setLogLevel("ERROR")

raw_trips = read_raw_trips(spark)
zones = read_zones(spark)

log_schema(raw_trips, "RAW HVFHV (before ETL)")

df = standardize_and_cast(raw_trips)
df = handle_nulls(df)
df = remove_duplicates(df)
df = clean_and_filter(df)
df = add_derived_fields(df)
df = enrich_with_zones(df, zones)
check_referential_integrity(df, zones)

# mark the DataFrame for reuse across the two actions that follow (SQL demo + write),
# avoiding recomputation of the whole chain twice.
df = df.cache()

log_schema(df, "CLEAN FACT_TRIP (after ETL)")
print(f"\nFinal row count: {df.count():,}")

run_sql_and_aggregation_demo(spark, df)
write_output(df)

df.unpersist()
spark.stop()

2026-09-02 21:29:32,254 WARN util.Utils: Your hostname, localhost.localdomain resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
2026-09-02 21:29:32,255 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
2026-09-02 21:29:37,687 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).



SCHEMA: RAW HVFHV (before ETL)
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true

Duplicate removal: 500,000 -> 500,000 rows (0 duplicates dropped)


Cleansing: writing 1 rejected rows to hdfs:///tlc/silver/trip_rejected_data



Referential integrity check: 0 pickup location IDs and 0 dropoff location IDs not found in the zones table.

SCHEMA: CLEAN FACT_TRIP (after ETL)
root
 |-- do_location_id: integer (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = false)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = false)
 |-- tolls: double (nullable = false)
 |-- bcf: double (nullable = false)
 |-- sales_tax: double (nullable = false)
 |-- congestion_surcharge: double (nullable = false)
 |-- airport_fee: double (nullable = false)
 |-- tips: double (nullable = false)
 |-- driver_pay: dou


Final row count: 499,999

--- Spark SQL demo: trips & avg fare by borough (top 10) ---
+-------------+---------------+----------+--------+----------------+
|pu_borough   |pu_service_zone|trip_count|avg_fare|avg_duration_sec|
+-------------+---------------+----------+--------+----------------+
|Manhattan    |Yellow Zone    |170218    |36.14   |1161.0          |
|Brooklyn     |Boro Zone      |130077    |24.8    |1066.0          |
|Queens       |Boro Zone      |84964     |24.01   |997.0           |
|Bronx        |Boro Zone      |60017     |21.81   |947.0           |
|Manhattan    |Boro Zone      |29097     |25.77   |1028.0          |
|Queens       |Airports       |18416     |75.37   |2115.0          |
|Staten Island|Boro Zone      |7189      |25.52   |937.0           |
|N/A          |N/A            |21        |35.71   |1108.0          |
+-------------+---------------+----------+--------+----------------+


--- DataFrame API aggregation demo: trips by day_of_week / peak hour ---
+--------


Wrote clean dataset to hdfs:///tlc/silver/trip_clean_data
